# P8. 법령 개정 비교기

개인정보 보호법 현행판과 시행예정판을 조문 번호로 맞춰 신설 · 삭제 · 변경 · 그대로를 가르고, 바뀐 조문은 항 · 호 단위 글자 diff(difflib)로 무엇이 바뀌었는지 뽑아 표(Parquet · DuckDB)로 남깁니다. 바뀐 조문만 ko-sroberta 로 임베딩해 Qdrant 에 넣고 「AI 관련해서 무엇이 바뀌었나?」에 두 판의 근거 조문을 나란히 보여 줍니다.

- 단계: 개별 프로젝트 · 비정형 · 예상 2~3일
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 법제처 국가법령정보 공동활용(DRF Open API) — 개인정보 보호법 현행판(법률 제21445호, 2026.3.10 공포 · 2026.9.11 시행)과 시행예정판(법률 제21910호, 2026.9.8 공포 · 2027.3.9 시행) 원문, 2026-10-28 받은 스냅샷(P4 와 같은 파일). 법령은 저작권법 제7조에 따라 보호받지 않는 저작물

셀을 위에서부터 차례로 실행하세요. 각 단계 끝의 **확인 포인트** 숫자가 같게 나오면 다음 단계로 갑니다.
따라하기를 한 번 끝냈다면 같은 내용의 [빈칸 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P8/P8_blank.ipynb)으로 다시 쳐 보세요.

법은 자주 바뀝니다. 개정 법률이 공포되면 법제처가 「신 · 구조문 대비표」를 내지만, 회사의 개인정보 담당자가 알고 싶은 것은 더 좁습니다 — **「우리 일과 관련된 조문 중 무엇이, 언제부터, 어떻게 바뀌나?」**

이 프로젝트에서는 두 판의 법령 원문만으로 그 대비표를 직접 만듭니다. 그리고 만든 결과를 7주차의 RAG 흐름에 얹어, 질문하면 **두 판의 조문을 나란히** 근거로 보여 주는 도구로 마무리합니다.

```text
두 판 JSON → 조문 표(조문 번호 열쇠) → 맞추기(신설 · 삭제 · 변경 · 그대로) → 항 · 호 단위 글자 diff(difflib)
  → 변경 표(Parquet · DuckDB) → 바뀐 조문만 임베딩(ko-sroberta) → Qdrant → 질문 → 두 판 나란히 (+ 키가 있으면 gpt-4o-mini 요약)
```

다 끝내면 다음 세 가지를 말할 수 있습니다.

- 조문을 **무엇으로** 맞춰야 하는지(순서도 글자도 아닌 번호 쌍) — 잘못 맞추면 어떤 가짜 변경이 생기는지
- 2027년 3월 9일 시행판에서 실제로 바뀌는 조문과, 그중 「개정 표시만 바뀐」 가짜 변경
- 변경 내역을 검색 가능한 근거로 만들어 질문에 답하는 방법

P4(법령 Q&A 챗봇)의 5단계에서 두 판을 간단히 비교했다면, 여기서는 그 비교를 제대로 된 도구로 만듭니다.

## 1단계 · 준비 — 라이브러리와 두 판의 법령

P4 와 같은 두 스냅샷을 씁니다. 2026-10-28 에 국가법령정보 공동활용 DRF API 로 받은 원문 JSON 입니다.

- `law_pipa_20260911.json.gz` — 개인정보 보호법 **현행판**(법률 제21445호, 2026.3.10 공포, 2026.9.11 시행), `lawService.do?OC=test&target=law&type=JSON&MST=283839`
- `law_pipa_20270309.json.gz` — 같은 법의 **시행예정판**(법률 제21910호, 2026.9.8 공포, 2027.3.9 시행), `MST=289415` (목록에서 찾으려면 `lawSearch.do` 에 `nw=2`)

법령은 저작권법 제7조(보호받지 못하는 저작물)에 해당해 자유롭게 이용할 수 있습니다. API 로 직접 받는 방법은 P4 의 2단계를 보세요.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")

import os, re, json, gzip, shutil, difflib, urllib.request
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

def load_law(name):
    with gzip.open(fetch(name), "rt", encoding="utf-8") as f:
        return json.load(f)["법령"]

cur = load_law("law_pipa_20260911.json.gz")       # 현행
nxt = load_law("law_pipa_20270309.json.gz")       # 시행예정
for law in (cur, nxt):
    b = law["기본정보"]
    print(b["법령명_한글"], "법률 제" + b["공포번호"] + "호", "공포", b["공포일자"], "시행", b["시행일자"])

개인정보 보호법 법률 제21445호 공포 20260310 시행 20260911
개인정보 보호법 법률 제21910호 공포 20260908 시행 20270309


> **확인 포인트** — `개인정보 보호법 법률 제21445호 공포 20260310 시행 20260911` 과 `… 제21910호 공포 20260908 시행 20270309` 두 줄이 나옵니다.

In [ ]:
assert (cur["기본정보"]["시행일자"], nxt["기본정보"]["시행일자"]) == ("20260911", "20270309")

## 2단계 · 조문 표 만들기 — 맞출 열쇠는 번호 쌍

조문 하나를 표 한 줄로 만듭니다. 맞출 **열쇠**는 `(조문번호, 조문가지번호)` 두 숫자입니다. 「제28조의12」는 `(28, 12)`, 「제28조」는 `(28, 0)` 입니다.
글은 조문 머리 · 항 · 호 · 목을 한 줄기로 이어 붙입니다(P4 8단계의 `law_text` 와 같은 순서).

In [ ]:
def as_list(x):
    """항 · 호 · 목이 dict 하나로 오거나 없을 때도 목록으로 맞춥니다(P4 4단계)."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def article_text(u):
    out = [u["조문내용"].strip()]
    for h in as_list(u.get("항")):
        if h.get("항내용"):
            out.append(h["항내용"].strip())
        for ho in as_list(h.get("호")):
            out.append(ho["호내용"].strip())
            out += [m["목내용"].strip() for m in as_list(ho.get("목"))]
    return "\n".join(out)

def article_table(law):
    rows = []
    for u in law["조문"]["조문단위"]:
        if u["조문여부"] != "조문":                         # 장 · 절 제목은 뺀다
            continue
        jo, br = int(u["조문번호"]), int(u.get("조문가지번호") or 0)
        rows.append({"jo": jo, "br": br, "label": f"제{jo}조" + (f"의{br}" if br else ""),
                     "title": u.get("조문제목", ""), "text": article_text(u)})
    return pd.DataFrame(rows)

A = article_table(cur)
B = article_table(nxt)
print("현행", len(A), "· 시행예정", len(B))
B.tail(3)

현행 127 · 시행예정 131


,jo,br,label,title,text
128,74,2,제74조의2,몰수ㆍ추징 등,제74조의2(몰수ㆍ추징 등) 제70조부터 제73조까지의 어느 하나에 해당하는 죄를 ...
129,75,0,제75조,과태료,제75조(과태료)\n① 다음 각 호의 어느 하나에 해당하는 자에게는 5천만원 이하의...
130,76,0,제76조,과태료에 관한 규정 적용의 특례,제76조(과태료에 관한 규정 적용의 특례) 제75조의 과태료에 관한 규정을 적용할 ...


> **확인 포인트** — 현행 **127**개, 시행예정 **131**개 조문입니다. 열쇠 `(jo, br)` 가 한 판 안에서 겹치지 않아야 맞출 수 있습니다.

In [ ]:
assert (len(A), len(B)) == (127, 131)
assert not A.duplicated(["jo", "br"]).any() and not B.duplicated(["jo", "br"]).any()

일부러 틀려 봅니다. 조문 번호를 **글자**(`label`)로 정렬하면 「제28조의12」가 어디에 놓일까요?

In [ ]:
by_text = sorted(B["label"])
by_num = B.sort_values(["jo", "br"])["label"].tolist()
i, j = by_text.index("제28조의12"), by_num.index("제28조의12")
print("글자 정렬:", by_text[i - 3:i + 5])
print("숫자 정렬:", by_num[j - 3:j + 2])
print("제28조의12 의 자리 — 글자:", i + 1, "번째 · 숫자:", j + 1, "번째")
print("제2조의 자리 — 글자:", by_text.index("제2조") + 1, "번째")

글자 정렬: ['제28조', '제28조의10', '제28조의11', '제28조의12', '제28조의13', '제28조의14', '제28조의15', '제28조의2']
숫자 정렬: ['제28조의9', '제28조의10', '제28조의11', '제28조의12', '제28조의13']
제28조의12 의 자리 — 글자: 29 번째 · 숫자: 59 번째
제2조의 자리 — 글자: 42 번째


> **확인 포인트** — 글자 정렬에서는 `제28조` 바로 뒤에 `제28조의10` 이 오고, `제28조의12` 는 **`제28조의2` 보다 앞**(전체 29번째)에 놓입니다. 숫자 정렬에서는 `제28조의9 · 제28조의10 · 제28조의11` 뒤, 전체 59번째가 제자리입니다. 제2조는 글자 정렬에서 **42번째**로 밀립니다.

In [ ]:
assert by_text.index("제28조의12") < by_text.index("제28조의2")
assert by_num.index("제28조의12") > by_num.index("제28조의9")
assert by_text.index("제2조") + 1 == 42 and (i + 1, j + 1) == (29, 59)

> **자주 겪는 오류** — 글자 비교는 왼쪽부터 한 글자씩 봅니다. `제28조의1` 까지 같으면 다음 글자 `2`(제28조의12)와 `조`가 끝난 자리(제28조의2 는 `2` 다음이 끝)를 비교하므로 `제28조의12` 가 먼저입니다. 숫자는 반드시 **숫자 열로** 정렬하세요(2주차 7장 MySQL 에서도 `VARCHAR` 번호 정렬이 같은 문제를 냅니다).

## 3단계 · 두 판 맞추기 — 신설 · 삭제 · 변경 · 그대로

두 표를 열쇠 `(jo, br)` 로 **완전 외부 조인**합니다. 한쪽에만 있으면 신설 또는 삭제, 양쪽에 다 있으면 글을 비교해 변경 또는 그대로입니다(2주차 7장 JOIN — `FULL OUTER JOIN`).

In [ ]:
M = A.merge(B, on=["jo", "br"], how="outer", suffixes=("_cur", "_next"), indicator=True)

def status(r):
    if r["_merge"] == "right_only":
        return "added"
    if r["_merge"] == "left_only":
        return "deleted"
    return "same" if r["text_cur"] == r["text_next"] else "changed"

M["status"] = M.apply(status, axis=1)
M["label"] = M["label_next"].fillna(M["label_cur"])
M = M.sort_values(["jo", "br"]).reset_index(drop=True)
print(M["status"].value_counts())
M[M["status"] != "same"][["label", "title_next", "status"]]

status
same       124
added        4
changed      3
Name: count, dtype: int64


,label,title_next,status
1,제2조,정의,changed
14,제7조의9,보호위원회의 심의ㆍ의결 사항 등,changed
58,제28조의12,인공지능기술 개발을 위한 개인정보의 이용,added
59,제28조의13,인공지능기술 개발을 위한 개인정보 처리에 대한 관리ㆍ감독,added
60,제28조의14,인공지능기술 개발을 위한 개인정보 처리 제한,added
61,제28조의15,이용 특례에 따른 적용의 일부 제외,added
79,제37조의2,자동화된 결정에 대한 정보주체의 권리 등,changed


> **확인 포인트** — **신설 4 · 변경 3 · 삭제 0 · 그대로 124** 입니다. 신설은 제28조의12 ~ 제28조의15(인공지능기술 개발을 위한 개인정보 이용 등), 변경은 제2조 · 제7조의9 · 제37조의2 — P4 5단계에서 본 것과 같습니다.

In [ ]:
cnt = M["status"].value_counts().to_dict()
assert cnt == {"same": 124, "added": 4, "changed": 3}
assert M.loc[M["status"] == "added", "label"].tolist() == ["제28조의12", "제28조의13", "제28조의14", "제28조의15"]
assert M.loc[M["status"] == "changed", "label"].tolist() == ["제2조", "제7조의9", "제37조의2"]

법령 JSON 에도 `조문변경여부` 칸이 있습니다. 우리가 찾은 7개(신설 4 + 변경 3)와 같은지 맞춰 봅니다 — **스스로 만든 판정을 다른 근거와 대조**하는 습관입니다.

In [ ]:
flag = {(int(u["조문번호"]), int(u.get("조문가지번호") or 0))
        for u in nxt["조문"]["조문단위"] if u["조문여부"] == "조문" and u.get("조문변경여부") == "Y"}
ours = set(M.loc[M["status"] != "same", ["jo", "br"]].itertuples(index=False, name=None))
print("법령 표시:", len(flag), "· 우리 판정:", len(ours), "· 같은가:", flag == ours)
assert flag == ours

법령 표시: 7 · 우리 판정: 7 · 같은가: True


이번에는 일부러 두 가지로 잘못 맞춰 봅니다.

In [ ]:
# (1) 순서로 맞추기 — 몇 번째 조문끼리 비교
by_pos = sum(a != b for a, b in zip(A.sort_values(["jo", "br"])["text"], B.sort_values(["jo", "br"])["text"]))
print("순서로 맞추면 바뀐 조문:", by_pos)

# (2) 조문 기록(JSON) 통째로 비교
ua = {(int(u["조문번호"]), int(u.get("조문가지번호") or 0)): u for u in cur["조문"]["조문단위"] if u["조문여부"] == "조문"}
ub = {(int(u["조문번호"]), int(u.get("조문가지번호") or 0)): u for u in nxt["조문"]["조문단위"] if u["조문여부"] == "조문"}
by_json = [k for k in ua if k in ub and json.dumps(ua[k], ensure_ascii=False) != json.dumps(ub[k], ensure_ascii=False)]
print("JSON 통째로 비교하면 바뀐 조문:", len(by_json), "— 예: 제15조에서 다른 칸",
      [f for f in ua[(15, 0)] if ua[(15, 0)][f] != ub[(15, 0)].get(f)])

순서로 맞추면 바뀐 조문: 71
JSON 통째로 비교하면 바뀐 조문: 127 — 예: 제15조에서 다른 칸 ['조문시행일자']


> **확인 포인트** — 순서로 맞추면 **71개**, JSON 통째로 비교하면 **127개 전부**가 「바뀐」 것으로 나옵니다 — 둘 다 가짜 변경입니다.
> 순서로 맞추면 제28조의12 ~ 15 네 조문이 끼어든 뒤부터 한 칸씩 밀려 엉뚱한 조문끼리 비교됩니다. JSON 통째로 비교하면 모든 조문의 `조문시행일자` 칸(20260911 → 20270309)이 달라서 내용과 상관없이 전부 다릅니다.

In [ ]:
assert by_pos == 71 and len(by_json) == 127

## 4단계 · 항 · 호 단위 diff — 무엇이 바뀌었나

조문 단위 비교는 「바뀌었다」까지만 알려 줍니다. 무엇이 바뀌었는지 보려면 한 단계 더 내려가 **항 · 호 한 줄씩** 맞춥니다. 열쇠는 `(jo, br, hang, ho)` 네 칸입니다.

In [ ]:
CIRC = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"

def unit_table(law):
    rows = []
    for u in law["조문"]["조문단위"]:
        if u["조문여부"] != "조문":
            continue
        jo, br = int(u["조문번호"]), int(u.get("조문가지번호") or 0)
        rows.append(dict(jo=jo, br=br, hang=0, ho="", text=u["조문내용"].strip()))      # 조문 머리(제목 · 본문)
        for h in as_list(u.get("항")):
            hn = (h.get("항번호") or "").strip()
            hang = CIRC.find(hn[:1]) + 1 if hn else 0
            if h.get("항내용"):
                rows.append(dict(jo=jo, br=br, hang=hang, ho="", text=h["항내용"].strip()))
            for ho in as_list(h.get("호")):
                hb = ho["호번호"].strip().rstrip(".") + (f"의{ho['호가지번호']}" if ho.get("호가지번호") else "")
                text = ho["호내용"].strip() + "".join("\n" + m["목내용"].strip() for m in as_list(ho.get("목")))
                rows.append(dict(jo=jo, br=br, hang=hang, ho=hb, text=text))
    return pd.DataFrame(rows)

KEY = ["jo", "br", "hang", "ho"]
UA, UB = unit_table(cur), unit_table(nxt)
U = UA.merge(UB, on=KEY, how="outer", suffixes=("_cur", "_next"))
print("항 · 호 줄 수 — 현행", len(UA), "· 시행예정", len(UB))
U["status"] = "same"
U.loc[U["text_cur"].isna(), "status"] = "added"
U.loc[U["text_next"].isna(), "status"] = "deleted"
both = U["text_cur"].notna() & U["text_next"].notna()
U.loc[both & (U["text_cur"] != U["text_next"]), "status"] = "changed"
U["status"].value_counts()

항 · 호 줄 수 — 현행 965 · 시행예정 996


status
same       962
added       31
changed      3
Name: count, dtype: int64

> **확인 포인트** — 현행 965줄 · 시행예정 996줄, 그중 **신설 31줄 · 변경 3줄**이고 나머지 962줄은 그대로입니다.

In [ ]:
assert (len(UA), len(UB)) == (965, 996)
assert U["status"].value_counts().to_dict() == {"same": 962, "added": 31, "changed": 3}

「변경 3줄」이 무엇이 바뀌었는지 글자 단위로 봅니다. `difflib.SequenceMatcher` 의 `get_opcodes()` 는 두 글을 같은 부분(equal)과 다른 부분(replace · delete · insert)으로 나눠 줍니다.

In [ ]:
def char_diff(a, b):
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return [(op, a[i1:i2], b[j1:j2]) for op, i1, i2, j1, j2 in sm.get_opcodes() if op != "equal"]

for _, r in U[U["status"] == "changed"].iterrows():
    print(f"제{r.jo}조" + (f"의{r.br}" if r.br else ""), f"항{r.hang}", char_diff(r.text_cur, r.text_next))

제2조 항0 [('insert', '', ', 2026.9.8')]
제7조의9 항1 [('insert', '', ', 2026.9.8')]
제37조의2 항1 [('delete', ' ', ''), ('insert', '', ' <개정 2026.9.8>')]


> **확인 포인트** — 셋 다 내용이 아니라 표시가 바뀌었습니다. 제2조 머리와 제7조의9제1항은 개정 표시 끝에 `, 2026.9.8` 이 붙었을 뿐이고, 제37조의2제1항은 `인공지능 기술` → `인공지능기술`(공백 하나 삭제)과 ` <개정 2026.9.8>` 추가입니다.

이것이 **가짜 변경**입니다. 개정 표시(`<개정 …>`)는 「이 자리가 언제 바뀌었다」는 기록이지 조문의 내용이 아닙니다. 표시를 지우고 다시 비교하고, 공백까지 없애고 한 번 더 비교해 변경의 **종류**를 가릅니다.

In [ ]:
TAG = re.compile(r"\s*<(개정|신설|전문개정|본조신설)[^>]*>")
def no_tag(s):
    return TAG.sub("", s) if isinstance(s, str) else s
def no_space(s):
    return re.sub(r"\s+", "", s) if isinstance(s, str) else s

def kind(r):
    if r["status"] != "changed":
        return r["status"]
    a, b = no_tag(r["text_cur"]), no_tag(r["text_next"])
    if a == b:
        return "tag_only"                     # 개정 표시만 바뀜 — 가짜 변경
    if no_space(a) == no_space(b):
        return "spacing"                      # 띄어쓰기만 바뀜
    return "content"                          # 내용이 바뀜

U["kind"] = U.apply(kind, axis=1)
U["kind"].value_counts()

kind
same        962
added        31
tag_only      2
spacing       1
Name: count, dtype: int64

> **확인 포인트** — `tag_only` 2줄(제2조 머리 · 제7조의9제1항), `spacing` 1줄(제37조의2제1항), 내용이 바뀐 `content` 는 0줄입니다. 이번 개정의 실질은 **신설 31줄**입니다.

In [ ]:
k = U["kind"].value_counts().to_dict()
assert k == {"same": 962, "added": 31, "tag_only": 2, "spacing": 1}

> **자주 겪는 오류** — 공백 차이를 무조건 「같다」로 버리면 안 됩니다. 제37조의2 의 `인공지능 기술` → `인공지능기술` 은 공백 하나지만, 법이 직접 `<개정 2026.9.8>` 을 붙인 **용어 정비**입니다(같은 개정에서 제2조제9호에 「인공지능기술」 정의가 새로 생겼습니다). 그래서 버리지 않고 `spacing` 이라는 종류로 남겨, 읽는 사람이 판단하게 합니다.

## 5단계 · 조문별로 묶어 보기

항 · 호 줄을 조문 단위로 묶어, 조문마다 「무엇이 몇 줄 바뀌었나」를 한 줄로 요약합니다. 신 · 구조문 대비표의 목차가 됩니다.

In [ ]:
U["label"] = [f"제{j}조" + (f"의{b}" if b else "") for j, b in zip(U["jo"], U["br"])]
U["unit"] = U["label"] + U["hang"].map(lambda h: f"제{h}항" if h else "") + U["ho"].map(lambda h: f"제{h}호" if h else "")

summary = (U[U["kind"] != "same"]
           .groupby(["jo", "br", "label"], as_index=False)
           .agg(units=("unit", "count"), kinds=("kind", lambda s: ",".join(sorted(set(s)))),
                first=("unit", "first"))
           .sort_values(["jo", "br"]))
summary = summary.merge(M[["jo", "br", "status", "title_next"]], on=["jo", "br"])
summary[["label", "title_next", "status", "kinds", "units"]]

,label,title_next,status,kinds,units
0,제2조,정의,changed,"added,tag_only",2
1,제7조의9,보호위원회의 심의ㆍ의결 사항 등,changed,"added,tag_only",3
2,제28조의12,인공지능기술 개발을 위한 개인정보의 이용,added,added,13
3,제28조의13,인공지능기술 개발을 위한 개인정보 처리에 대한 관리ㆍ감독,added,added,5
4,제28조의14,인공지능기술 개발을 위한 개인정보 처리 제한,added,added,9
5,제28조의15,이용 특례에 따른 적용의 일부 제외,added,added,1
6,제37조의2,자동화된 결정에 대한 정보주체의 권리 등,changed,spacing,1


> **확인 포인트** — 7개 조문이 나옵니다. 제28조의12 가 13줄로 가장 많고, 제2조는 `added,tag_only`(제9호 「인공지능기술」 정의 신설 + 개정 표시), 제7조의9 는 `added,tag_only`(제1항제6호의3 · 6호의4 신설), 제37조의2 는 `spacing` 입니다.

In [ ]:
s = summary.set_index("label")
assert len(summary) == 7 and s["units"].idxmax() == "제28조의12" and s.loc["제28조의12", "units"] == 13
assert s.loc["제2조", "kinds"] == "added,tag_only" and s.loc["제37조의2", "kinds"] == "spacing"
assert U.loc[(U.jo == 2) & (U.ho == "9"), "text_next"].iloc[0].startswith('9. "인공지능기술"이란')

## 6단계 · 결과를 표로 남기기 — Parquet 과 DuckDB

비교 결과는 다음 사람이 다시 계산하지 않도록 표로 남깁니다. 조문 표(`M`)와 항 · 호 표(`U`)를 Parquet 으로 저장하고(3주차 13장), DuckDB 로 다시 읽어 질문합니다(3주차 14장).
`diff` 열에는 4단계의 글자 diff 를 JSON 글로 넣어 둡니다.

In [ ]:
U["diff"] = [json.dumps(char_diff(a, b), ensure_ascii=False) if k in ("tag_only", "spacing", "content") else None
             for a, b, k in zip(U["text_cur"], U["text_next"], U["kind"])]
meta = {"law": "개인정보 보호법", "cur": cur["기본정보"]["공포번호"], "next": nxt["기본정보"]["공포번호"],
        "next_effective": nxt["기본정보"]["시행일자"]}
for df_, name in [(M.drop(columns=["_merge"]), "p8_pipa_articles"), (U, "p8_pipa_units")]:
    out = df_.assign(**meta)
    out.to_parquet(f"data/{name}.parquet", index=False)
    print(name, len(out), "줄", round(os.path.getsize(f"data/{name}.parquet") / 1e3), "KB")

p8_pipa_articles 131 줄 148 KB
p8_pipa_units 996 줄 156 KB


In [ ]:
q = duckdb.sql("""
    SELECT label, unit, kind, left(text_next, 50) AS text_next
    FROM 'data/p8_pipa_units.parquet'
    WHERE kind <> 'same' AND text_next LIKE '%인공지능%'
    ORDER BY jo, br, hang, ho
""").df()
print(len(q), "줄")
q.head(8)

13 줄


,label,unit,kind,text_next
0,제2조,제2조제9호,added,"9. ""인공지능기술""이란 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」 제2..."
1,제7조의9,제7조의9제1항제6의3호,added,6의3. 제28조의12제1항 및 제2항에 따른 인공지능기술 개발을 위한 개인정보의 ...
2,제7조의9,제7조의9제1항제6의4호,added,6의4. 제28조의14제1항에 따른 인공지능기술 개발을 위한 개인정보 처리 제한에 ...
3,제28조의12,제28조의12,added,제28조의12(인공지능기술 개발을 위한 개인정보의 이용)
4,제28조의12,제28조의12제1항,added,① 개인정보처리자는 인공지능기술 개발(성능 개선을 포함한다. 이하 이 절에서 같다)...
5,제28조의12,제28조의12제1항제1호,added,"1. 영상, 음성, 이미지, 부호 및 문자 등 처리되는 정보의 특성, 인공지능기술 ..."
6,제28조의12,제28조의12제1항제3호,added,3. 인공지능기술 개발의 목적이 다음 각 목의 어느 하나를 포함하고 있는 경우로서 ...
7,제28조의12,제28조의12제4항,added,④ 보호위원회는 제2항에 따른 심의ㆍ의결을 요청한 내용이 종전에 보호위원회의 심의ㆍ...


> **확인 포인트** — 바뀐 줄 34줄 중 「인공지능」이 든 줄이 **13줄**입니다. 숫자 열 `jo, br, hang` 으로 정렬해서 제2조제9호 → 제7조의9 → 제28조의12 … 순서로 나옵니다.

In [ ]:
assert len(q) == 13 and q.iloc[0]["unit"] == "제2조제9호"

> **Aebon's Tip** — 법이 바뀔 때마다 이 표에 판(공포번호) 열을 두고 행을 쌓아 가면, 「제28조의12 는 언제 생겼나」 같은 이력 질문을 SQL 하나로 답할 수 있습니다. 매일 시행예정 목록(`nw=2`)을 확인해 새 판이 보이면 이 노트북을 돌리는 일은 8주차 Airflow DAG 로 만들기 좋습니다.

## 7단계 · 바뀐 조문만 임베딩해 Qdrant 에 넣기

질문은 「AI 관련해서 무엇이 바뀌었나?」처럼 들어옵니다. 그런데 법령에는 「AI」가 없습니다. 먼저 글자 검색으로 확인합니다.

In [ ]:
n_ai = duckdb.sql("SELECT count(*) FROM 'data/p8_pipa_units.parquet' WHERE text_next ILIKE '%AI%'").fetchall()[0][0]
print("법령에 「AI」가 든 줄:", n_ai)
assert n_ai == 0

법령에 「AI」가 든 줄: 0


글자가 다른 말을 뜻으로 이으려면 임베딩이 필요합니다(7주차 7-2장). 전체 996줄을 다 넣지 않고 **바뀐 줄만** 넣습니다. 「무엇이 바뀌었나」라는 질문의 답은 바뀐 줄 안에만 있고, 그대로인 줄이 섞이면 답과 상관없는 조문이 위로 올라옵니다.
개정 표시만 바뀐 `tag_only` 줄은 내용이 같으니 넣지 않습니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

docs = U[U["kind"].isin(["added", "spacing", "content"])].copy()
title = M.set_index(["jo", "br"])["title_next"]
docs["doc"] = [f"{l}({title[(j, b)]}) {no_tag(t)}" for l, j, b, t in zip(docs["label"], docs["jo"], docs["br"], docs["text_next"])]
print(len(docs), "줄을 넣습니다")

model = SentenceTransformer("jhgan/ko-sroberta-multitask")
qd = QdrantClient(":memory:")
vecs = model.encode(docs["doc"].tolist(), batch_size=32, normalize_embeddings=True)
qd.create_collection("pipa_changes", vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE))
qd.upsert("pipa_changes", [
    models.PointStruct(id=i, vector=v.tolist(), payload={
        "unit": r.unit, "label": r.label, "jo": int(r.jo), "br": int(r.br), "kind": r.kind,
        "cur": r.text_cur if isinstance(r.text_cur, str) else None, "next": r.text_next})
    for i, (v, r) in enumerate(zip(vecs, docs.itertuples()))])
print("Qdrant:", qd.count("pipa_changes").count)

32 줄을 넣습니다


Qdrant: 32


> **확인 포인트** — 신설 31줄 + 띄어쓰기 1줄 = **32줄**이 Qdrant 에 들어갑니다(Colab CPU 에서 모델 내려받기 포함 1분 안쪽).

In [ ]:
assert len(docs) == 32 and qd.count("pipa_changes").count == 32

## 8단계 · 질문하고 두 판을 나란히 보기

검색 결과는 바뀐 항 · 호입니다. 사람이 판단하려면 **그 조문 전체를 두 판으로 나란히** 봐야 하므로, 1위부터 조문 번호를 모아 두 판의 조문 글(`A` · `B`)을 함께 보여 줍니다.

In [ ]:
def ask(q, k=5):
    qv = model.encode(q, normalize_embeddings=True).tolist()
    return qd.query_points("pipa_changes", query=qv, limit=k).points

def side_by_side(hits, n_articles=3):
    seen = []
    for h in hits:
        key = (h.payload["jo"], h.payload["br"])
        if key not in seen:
            seen.append(key)
    out = []
    for jo, br in seen[:n_articles]:
        row = M[(M.jo == jo) & (M.br == br)].iloc[0]
        units = [(h.payload["unit"], h.payload["cur"] or "(없음 — 신설)", h.payload["next"])
                 for h in hits if (h.payload["jo"], h.payload["br"]) == (jo, br)]
        out.append({"label": row.label, "status": row.status, "hit_units": units,
                    "current": row.text_cur if isinstance(row.text_cur, str) else "(현행판에 없음 — 신설)",
                    "next": row.text_next})
    return out

q = "AI 관련해서 무엇이 바뀌었나?"
hits = ask(q)
for h in hits:
    print(f"{h.score:.3f}  {h.payload['unit']:14s} {h.payload['kind']:8s} {h.payload['next'][:40]}")

0.517  제2조제9호         added    9. "인공지능기술"이란 「인공지능 발전과 신뢰 기반 조성 등에 관한 기
0.394  제7조의9제1항제6의3호  added    6의3. 제28조의12제1항 및 제2항에 따른 인공지능기술 개발을 위한 
0.390  제7조의9제1항제6의4호  added    6의4. 제28조의14제1항에 따른 인공지능기술 개발을 위한 개인정보 처
0.383  제28조의12제4항     added    ④ 보호위원회는 제2항에 따른 심의ㆍ의결을 요청한 내용이 종전에 보호위원
0.377  제28조의13        added    제28조의13(인공지능기술 개발을 위한 개인정보 처리에 대한 관리ㆍ감독)


> **확인 포인트** — 1위는 **제2조제9호**(「인공지능기술」 정의 신설, 0.517), 그다음 제7조의9제1항제6호의3 · 6호의4(보호위원회 심의 · 의결 사항에 인공지능 관련 항목 추가)가 나옵니다. 질문의 「AI」가 법령의 「인공지능」으로 이어졌습니다.

In [ ]:
assert hits[0].payload["unit"] == "제2조제9호" and round(hits[0].score, 3) == 0.517
assert [h.payload["unit"] for h in hits[1:3]] == ["제7조의9제1항제6의3호", "제7조의9제1항제6의4호"]

In [ ]:
for a in side_by_side(hits):
    print("=" * 70)
    print(a["label"], f"[{a['status']}]", f"조문 길이 현행 {len(a['current'])}자 → 시행예정 {len(a['next'])}자")
    for unit, before, after in a["hit_units"]:
        print(f"  {unit}")
        print(f"    현행(2026.9.11)   : {no_tag(before)[:70]}")
        print(f"    시행예정(2027.3.9): {no_tag(after)[:70]}")

제2조 [changed] 조문 길이 현행 1331자 → 시행예정 1409자
  제2조제9호
    현행(2026.9.11)   : (없음 — 신설)
    시행예정(2027.3.9): 9. "인공지능기술"이란 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」 제2조제3호에 따른 인공지능기술을 말한다.
제7조의9 [changed] 조문 길이 현행 1008자 → 시행예정 1126자
  제7조의9제1항제6의3호
    현행(2026.9.11)   : (없음 — 신설)
    시행예정(2027.3.9): 6의3. 제28조의12제1항 및 제2항에 따른 인공지능기술 개발을 위한 개인정보의 이용에 관한 사항
  제7조의9제1항제6의4호
    현행(2026.9.11)   : (없음 — 신설)
    시행예정(2027.3.9): 6의4. 제28조의14제1항에 따른 인공지능기술 개발을 위한 개인정보 처리 제한에 관한 사항
제28조의12 [added] 조문 길이 현행 14자 → 시행예정 1401자
  제28조의12제4항
    현행(2026.9.11)   : (없음 — 신설)
    시행예정(2027.3.9): ④ 보호위원회는 제2항에 따른 심의ㆍ의결을 요청한 내용이 종전에 보호위원회의 심의ㆍ의결을 거친 인공지능 기술ㆍ서비스의 내용ㆍ방


> **확인 포인트** — 제2조 · 제7조의9 · 제28조의12 세 조문이 나옵니다. 세 조문 모두 검색된 줄의 현행 칸이 「(없음 — 신설)」이고, 제28조의12 는 조문 전체가 현행판에 없습니다(「(현행판에 없음 — 신설)」).

In [ ]:
arts = side_by_side(hits)
assert [a["label"] for a in arts] == ["제2조", "제7조의9", "제28조의12"]
assert arts[2]["current"].startswith("(현행판에 없음")

일부러 다른 질문도 던져 봅니다. **이번 개정과 상관없는 질문**에도 Qdrant 는 늘 무언가를 1위로 돌려줍니다.

In [ ]:
off = ask("CCTV 설치 안내판에는 무엇을 적어야 하나요?")
print(round(off[0].score, 3), off[0].payload["unit"], off[0].payload["next"][:50])
print("AI 질문 1위 점수:", round(hits[0].score, 3))

0.301 제28조의14제3항 ③ 제1항 및 제2항에서 규정한 사항 외에 개인정보 처리 제한의 세부 기준 및 절차 등에 
AI 질문 1위 점수: 0.517


> **확인 포인트** — CCTV 질문에도 결과가 나오지만 1위 점수가 **0.301**(제28조의14제3항)로, AI 질문(0.517)보다 낮습니다. 「바뀐 조문 중에는 관련된 것이 없다」고 답해야 할 질문입니다.

In [ ]:
assert off[0].score < 0.4

> **자주 겪는 오류** — 벡터 검색은 「없음」을 모릅니다. 점수 기준(여기서는 0.45 쯤)을 질문 여러 개로 재서 정하고, 그 아래면 「이번 개정에서 바뀐 조문 중에는 관련된 것이 없습니다(현행 조문은 P4 챗봇에서 찾으세요)」라고 답하게 하세요.

## 9단계 · 답 생성 — 두 판의 근거로 요약

검색된 조문의 **현행 · 시행예정 두 글과 글자 diff** 를 프롬프트에 넣고, 「바뀐 점만, 조문 번호를 붙여 요약해줘」라고 요청합니다. OpenAI 키가 있을 때만 gpt-4o-mini 를 부르고, 없으면 8단계의 두 판 나란히 보기로 끝납니다(Colab 이면 왼쪽 열쇠 아이콘 「보안 비밀」에 `OPENAI_API_KEY`).

In [ ]:
KEY = os.environ.get("OPENAI_API_KEY")
if not KEY:
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = None

def build_prompt(q, hits):
    parts = []
    for h in hits:
        p = h.payload
        before = no_tag(p["cur"]) if p["cur"] else "(없음 — 신설)"
        parts.append(f"[{p['unit']}] 종류: {p['kind']}\n현행: {before}\n시행예정: {no_tag(p['next'])}")
    ctx = "\n\n".join(parts)
    return (f"다음은 개인정보 보호법 현행판(2026.9.11 시행)과 시행예정판(법률 제21910호, 2027.3.9 시행)의 바뀐 조문입니다.\n\n{ctx}\n\n"
            f"질문: {q}\n"
            "위 자료 안에서만 근거를 찾아, 바뀌는 점을 3~5개 항목으로 요약해줘. 항목마다 [제○조…] 형식으로 조문을 붙이고, "
            "2027년 3월 9일부터 시행된다는 점을 첫 줄에 적어줘. 자료에 없는 내용은 쓰지 말아줘.")

prompt = build_prompt(q, hits)
if KEY:
    need("openai")
    from openai import OpenAI
    res = OpenAI(api_key=KEY).chat.completions.create(
        model="gpt-4o-mini", temperature=0, messages=[{"role": "user", "content": prompt}])
    print(res.choices[0].message.content)
else:
    print("OpenAI 키가 없어 요약은 건너뜁니다. 프롬프트에 들어간 근거:")
    print(prompt[:900], "…")

OpenAI 키가 없어 요약은 건너뜁니다. 프롬프트에 들어간 근거:
다음은 개인정보 보호법 현행판(2026.9.11 시행)과 시행예정판(법률 제21910호, 2027.3.9 시행)의 바뀐 조문입니다.

[제2조제9호] 종류: added
현행: (없음 — 신설)
시행예정: 9. "인공지능기술"이란 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」 제2조제3호에 따른 인공지능기술을 말한다.

[제7조의9제1항제6의3호] 종류: added
현행: (없음 — 신설)
시행예정: 6의3. 제28조의12제1항 및 제2항에 따른 인공지능기술 개발을 위한 개인정보의 이용에 관한 사항

[제7조의9제1항제6의4호] 종류: added
현행: (없음 — 신설)
시행예정: 6의4. 제28조의14제1항에 따른 인공지능기술 개발을 위한 개인정보 처리 제한에 관한 사항

[제28조의12제4항] 종류: added
현행: (없음 — 신설)
시행예정: ④ 보호위원회는 제2항에 따른 심의ㆍ의결을 요청한 내용이 종전에 보호위원회의 심의ㆍ의결을 거친 인공지능 기술ㆍ서비스의 내용ㆍ방식ㆍ형태 등과 실질적으로 동일하거나 유사한 경우로서 대통령령으로 정하는 경우에는 보호위원회가 정하여 고시하는 방법에 따라 심의ㆍ의결 절차를 간소화할 수 있다.

[제28조의13] 종류: added
현행: (없음 — 신설)
시행예정: 제28조의13(인공지능기술 개발을 위한 개인정보 처리에 대한 관리ㆍ감독)

질문: AI 관련해서 무엇이 바뀌었나?
위 자료 안에서만 근거를 찾아, 바뀌는 점을 3~5개 항목으로 요약해줘. 항목마다 [제○조…] 형식으로 조문을 붙이고, 2027년 3월 9일부터 시행된다는 점을 첫 줄에 적어줘. 자료에 없는 내용은 쓰지 말아줘. …


> **확인 포인트** — 키가 없으면 프롬프트 앞부분에 `[제2조제9호] 종류: added` 와 `현행: (없음 — 신설)` 이 보입니다. 키가 있으면 답의 항목마다 `[제2조제9호]` · `[제28조의12…]` 같은 인용이 붙는지 확인하세요.

In [ ]:
assert "[제2조제9호] 종류: added" in prompt and "현행: (없음 — 신설)" in prompt
assert "<개정" not in prompt                    # 가짜 변경의 원인인 개정 표시는 프롬프트에 넣지 않는다

> **Aebon's Tip** — 답에서 `\[제\d+조[^\]]*\]` 패턴을 뽑아 `hits` 의 `unit` 에 없는 번호가 있으면 경고하세요(P4 9단계와 같은 인용 검증). 개정 비교는 특히 「현행에도 있던 내용을 새로 생긴 것처럼」 요약하는 실수가 잦으니, 항목마다 `kind` 가 `added` 인지도 함께 확인하면 좋습니다.

## 10단계 · 더 해 보기

- 국가법령정보 API 의 `lsHistory`(법령 연혁)로 개인정보 보호법의 이전 판을 하나 더 받아, 판 세 개를 차례로 비교해 「제28조의2 는 언제 생겼나」를 6단계 표로 답해 보세요.
- 「개인정보 보호법 시행령」의 현행 · 시행예정 두 판으로 같은 노트북을 돌려 보세요. 시행령은 법보다 늦게 바뀌는 일이 많아 「법은 바뀌었는데 시행령은 아직」인 조문을 찾을 수 있습니다.
- 조문이 **옮겨진** 경우(예: 제39조의3 이 제39조의5 로)는 번호로 맞추면 「삭제 1 + 신설 1」로 보입니다. `조문이동이전` · `조문이동이후` 칸을 써서 이동을 따로 가려 보세요.
- Streamlit(8주차)으로 왼쪽에 현행, 오른쪽에 시행예정을 놓고 diff 부분에 색을 칠하는 대비표 화면을 만들어 보세요(`difflib.HtmlDiff` 가 출발점이 됩니다).